# Lesson 46 · The mel scale — the ear is a logarithmic machine: the mel frequency scale and the triangular mel filterbank

**Goal**: first understand why the ear "hears" frequency on an uneven ruler, then implement Hz↔mel and the triangular filterbank yourself.

> Plain words before the formula: mel doesn't make Hz more complicated; it squeezes it into a ruler that's closer to how we hear. Whisper typically uses about **80** mel bands — later, in L47, they get wired into the log-mel input.

🔗 Aurora connection: `aurora.audio.mel.hz_to_mel()` and `aurora.audio.mel.mel_filterbank()`

← **Previous lesson** · [L45 · Making spectrograms](L45_spectrogram.ipynb)

> Last lesson covered **making spectrograms**: from the |STFT|² power spectrum to a dB heatmap — an X-ray of sound in time and frequency.
> This lesson explores **the mel scale**.

## This lesson's story: why isn't the piano keyboard evenly spaced in frequency?

A piano's 88 keys span frequencies from 27 Hz in the bass to 4186 Hz in the treble. Look closely, though, and you'll notice that neighboring keys in the bass are only a few hertz apart, while neighboring keys in the treble can differ by a couple of hundred hertz. That's not lazy keyboard design — the ear simply doesn't hear the world on a linear ruler.

**The ear is a logarithmic sensor**: you can tell 100 Hz from 110 Hz (10 Hz apart), but you can barely hear the difference between 3000 Hz and 3010 Hz (also 10 Hz apart). To the ear, "one octave higher" means the frequency doubles — a multiplicative relationship, not an additive one.

The mel scale (1937) captures this logarithmic perception, and today it's usually written as a formula:

$$\text{mel}(f) = 2595 \cdot \log_{10}\!\left(1 + \frac{f}{700}\right)$$

**Key values**:
- 1000 Hz → 1000 mel (the reference point)
- 2000 Hz → 1521 mel (the frequency goes up 1000 Hz, the mel value only 521)
- 4000 Hz → 2146 mel (the frequency goes up another 2000 Hz, and the mel value only 625)

See the pattern? The step in Hz doubled (1000 → 2000), yet the step in mel barely changed. The higher the frequency, the more flatly the same number of Hz gets squashed on the mel axis — that's logarithmic compression, and it matches the ear's "less sensitive at high frequencies" perception.

**The mel filterbank**: think of it as a row of triangular windows, first laid out evenly on the mel axis and then projected back onto the Hz axis. After projection they look denser at low frequencies and sparser at high frequencies, which mirrors how the cochlea resolves different frequencies. This is also the first step of MFCC features.

Task for this section: implement `hz_to_mel(f)` and `mel_filterbank(n_mels, n_fft, sr)`.

## A visual bridge: piano keys → frequency → mel → cochlea

Why "fine at low frequencies, coarse at high frequencies"? Follow this chain down and the intuition arrives before the formula (the formula is in the next section):

```
Piano keyboard (keys equally wide, steps that sound equal)
   │  each octave up = physical frequency doubles (×2, not + a fixed amount)
   ▼
Physical frequency   27 → 55 → 110 → 220 → 440 → 880 → 1760 Hz   (octaves get wider to the right)
   │  hz_to_mel(): straightens the "doubling" multiplication into addition
   ▼
Mel axis     |--|--|--|--|--|--|   octaves become (roughly) evenly spaced here
   │  cut evenly into N pieces, then project back to Hz
   ▼
Basilar membrane  dense at low frequencies ‖‖‖‖‖ …… sparse at high frequencies ‖   ‖     ‖
```

In one sentence: the ear perceives pitch by **ratios** (logarithmically), and the mel axis stretches that "sense of ratio" into "equal distances". So back on the Hz axis, the low-frequency filters are narrow and dense and the high-frequency filters are wide and sparse — which fits the physical layout of the basilar membrane in the cochlea.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The mel scale formula

The mel scale was proposed by Stevens, Volkmann and Newman (1937); the commonly used approximation (the HTK formula, which is Aurora's default) is:

```
f_mel = 2595 * log10(1 + f_hz / 700)
```

A 700 Hz tone corresponds to about 781 mel. Note:
- low range (0–1000 Hz): about 100 mel per 100 Hz on average (more near 0 Hz), high resolution
- high range (3000–4000 Hz): about 30 mel per 100 Hz, low resolution

The inverse (mel → Hz):
```
f_hz = 700 * (10^(f_mel / 2595) - 1)
```

### Derivation: the inverse transform, mel → Hz

Start from the forward formula:
$$\text{mel} = 2595 \cdot \log_{10}\!\left(1 + \frac{f}{700}\right)$$

To solve for $f$, go step by step:

**Step 1**: divide both sides by 2595
$$\frac{\text{mel}}{2595} = \log_{10}\!\left(1 + \frac{f}{700}\right)$$

**Step 2**: raise 10 to the power of both sides (since this is log₁₀, the inverse operation is 10^x)
$$10^{\frac{\text{mel}}{2595}} = 1 + \frac{f}{700}$$

**Step 3**: subtract 1 from both sides
$$10^{\frac{\text{mel}}{2595}} - 1 = \frac{f}{700}$$

**Step 4**: multiply both sides by 700
$$f = 700 \cdot \left(10^{\frac{\text{mel}}{2595}} - 1\right)$$

Check: plugging in mel=1000 should give f=1000 Hz
$$f = 700 \cdot (10^{1000/2595} - 1) = 700 \cdot (10^{0.3855} - 1) = 700 \cdot (2.4286 - 1) = 700 \cdot 1.4286 \approx 1000 \text{ Hz} \checkmark$$

## 1.5 The story behind the mel formula: why 2595 and 700?

In 1937, Stevens, Volkmann and Newman ran a key experiment: listeners heard two pure tones and adjusted the frequency of the second until the "perceived distances" came out equal.

**The result**: perceived distance grows in proportion to **log frequency** rather than linear frequency. Specifically, they found:
- people's resolution is highest around 100 Hz (they can hear the smallest frequency differences)
- around 1 kHz the resolution starts to fall
- at higher frequencies it keeps falling

An empirical formula was then fitted to curves like these (the 2595/700 form in use today was popularized later, notably by O'Shaughnessy in 1987 and the HTK toolkit). The reference point is chosen as **1000 Hz = 1000 mel** (the name comes from the root of "melody"), which keeps the formula tidiest around 1 kHz.

The fit that's used today is:
$$\text{mel}(f) = 2595 \cdot \log_{10}\!\left(1 + \frac{f}{700}\right)$$

where:
- **2595** is a scale factor chosen so that 1000 Hz comes out at exactly 1000 mel (the reference point)
- **700** is a corner frequency that controls how strongly low and high frequencies are compressed — it reflects how the ear's resolution changes across frequency

A quick check: plug in the reference point
$$\text{mel}(1000) = 2595 \cdot \log_{10}(1 + 1000/700) = 2595 \cdot \log_{10}(1700/700) = 2595 \cdot \log_{10}(2.4286)$$

Let's actually compute it: $\log_{10}(1 + 1000/700) = \log_{10}(2.4286) \approx 0.3853$, and $0.3853 \cdot 2595 \approx 1000$ ✓

> 💡 **Key insight**: 2595 and 700 aren't "magic numbers" pulled from thin air; they're parameters fitted to real listening-experiment data. Together they make the formula perceptually accurate.

In [ ]:
# Demo: the logarithmic character of the mel scale
freqs = np.array([100, 200, 500, 1000, 2000, 4000, 8000])
mels_approx = 2595 * np.log10(1 + freqs / 700)

print(f"{'Hz':>8}  {'Mel':>8}  {'Delta_Mel':>10}")
for i, (f, m) in enumerate(zip(freqs, mels_approx)):
    delta = f"{mels_approx[i] - mels_approx[i-1]:.1f}" if i > 0 else "—"
    print(f"{f:>8}  {m:>8.1f}  {delta:>10}")

## 2. Building the triangular filterbank

The steps:
1. In the mel domain, take `n_mels + 2` evenly spaced points from 0 to `hz_to_mel(sr/2)` (including the left and right edges)
2. Convert these `n_mels + 2` mel values back to Hz, giving unevenly spaced center frequencies
3. Map each Hz center frequency to an FFT bin index (rounding down)
4. Triangular filter `m`: rises linearly to 1 over `[center[m-1], center[m]]` and falls linearly to 0 over `[center[m], center[m+1]]`

In the Hz domain, each filter is an asymmetric triangle — narrow at low frequencies, wide at high frequencies, a natural match for the ear's resolution.

## 2.5 Why triangular filters?

Before building the triangular filters, ask a question: why not a **rectangle** (all 1 or all 0), a **trapezoid**, or some other shape?

| Shape | Definition | Pros | Cons |
|------|------|------|------|
| **Rectangle** | 1 inside [center-w, center+w], 0 elsewhere | simplest | steep edges, high-frequency confusion (spectral leakage) |
| **Triangle** | rises linearly from 0 to 1, then falls back to 0 | smooth edges, less confusion; close to the ear's response | slightly more computation |
| **Trapezoid** | a flat top with sloping sides | flat frequency response on top, more like a real filter | more parameters, harder to standardize |
| **Hamming/Hann window** | a cosine shape | especially smooth | more complex to compute, less friendly to standardization |

**Why did MFCC pick triangles?**
1. **The shape approximates cochlear filters**: the real filters along the basilar membrane (cochlear filters) are roughly symmetric triangles (or bells) on a log-frequency axis
2. **Simple to compute**: just three anchor points and linear interpolation, easy to compute and check by hand
3. **Balance**: not as crude as a rectangle, not as complex as a cosine window — a balance of practicality and accuracy

In one sentence: triangular filters became the standard choice for MFCC because they have **smooth edges, are simple to compute, and have a clear physical meaning**.

In [ ]:
# Visualization: even in the mel domain → uneven in the Hz domain
sr = 16000
n_mels = 10
mel_max = 2595 * np.log10(1 + (sr / 2) / 700)
mel_points = np.linspace(0, mel_max, n_mels + 2)
hz_points = 700 * (10 ** (mel_points / 2595) - 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(mel_points, np.zeros(n_mels+2), 'o')
axes[0].set_title('Mel domain: evenly spaced')
axes[0].set_xlabel('Mel')
axes[1].plot(hz_points, np.zeros(n_mels+2), 'o')
axes[1].set_title('Hz domain: log spacing (dense low, sparse high)')
axes[1].set_xlabel('Hz')
plt.tight_layout()
plt.show()

## 3. Multiplying the filterbank matrix by the power spectrum

`mel_filterbank()` returns a matrix of shape `(n_mels, n_fft//2+1)`; each row holds one triangular filter's weights along the FFT-bin axis.

```
fb = mel_filterbank(40, 512, 16000)   # shape (40, 257)
power_spec = np.abs(X)**2             # shape (257,), one frame's power spectrum
mel_energy = fb @ power_spec          # shape (40,), mel energies
```

A single matrix multiplication `@` does all 40 triangular integrals. Take the log of `mel_energy` and you have log-mel features; apply a DCT and you have MFCCs (L49-L50).

In [ ]:
# Demo of the matrix multiplication: real triangular filter weights -> mel energies
# (a simplified version for illustration; the full filterbank is implemented after the coding task)

# For the demo, we build the real triangular weights of the first 5 filters by hand
# Assume sr=16000, n_fft=512, n_bins=257
np.random.seed(42)
sr, n_fft, n_bins = 16000, 512, 257

# Build 5 mel filters (simplified demo)
n_demo_filters = 5
mel_min, mel_max = 0, 2595 * np.log10(1 + sr/2 / 700)
mel_centers = np.linspace(mel_min, mel_max, n_demo_filters + 2)
hz_centers = 700 * (10 ** (mel_centers / 2595) - 1)
bin_centers = np.floor((n_fft + 1) * hz_centers / sr).astype(int)

# Build the real triangular filter matrix
fb_demo = np.zeros((n_demo_filters, n_bins))
for m in range(n_demo_filters):
    lo, ctr, hi = bin_centers[m], bin_centers[m+1], bin_centers[m+2]
    # rising edge
    for k in range(lo, ctr):
        if ctr > lo:
            fb_demo[m, k] = (k - lo) / (ctr - lo)
    # falling edge
    for k in range(ctr, hi):
        if hi > ctr:
            fb_demo[m, k] = (hi - k) / (hi - ctr)

# Use a meaningful power spectrum for the demo: strong at low frequencies, weak at high
power_spectrum = 1.0 / (1 + np.arange(n_bins) / 30)  # a simplified spectral envelope

mel_energy_demo = fb_demo @ power_spectrum
print(f"Real triangular filter demo")
print(f"  power spectrum shape: {power_spectrum.shape}")
print(f"  filterbank shape: {fb_demo.shape}")
print(f"  mel_energy shape: {mel_energy_demo.shape}")
print(f"  mel_energy (filters from low to high): {mel_energy_demo}")
print(f"\nNotice: the low-frequency filter captures more energy (the power spectrum is strong at low frequencies), and the high-frequency filters capture less — as expected.")

## 4. ✏️ Implement `hz_to_mel(f)`

**Just apply the formula**: $\text{mel}(f) = 2595 \cdot \log_{10}(1 + f/700)$

| Implementation hint | Code |
|---|---|
| a single frequency | `2595 * np.log10(1 + f / 700)` |
| array input | `np.log10` broadcasts, so it just works |

**Values to check against** (computed by hand):
- `hz_to_mel(0)` = 0.0 (0 Hz = 0 mel)
- `hz_to_mel(700)` ≈ 781.2 mel
- `hz_to_mel(1000)` ≈ 1000.0 mel (the reference point)
- `hz_to_mel(4000)` ≈ 2146.1 mel

> Inverse: `mel_to_hz(m) = 700 * (10^(m/2595) - 1)`

### Tip: NumPy broadcasting

When implementing `hz_to_mel(f)`, we want it to handle both a single number and an array. Python's built-in functions usually can't, but NumPy's functions can.

**Broadcasting** is a core NumPy feature: when you apply an operation to an array, NumPy applies it **element by element**.

Example:
```python
f = np.array([100, 500, 1000, 4000])
result = 2595 * np.log10(1 + f / 700)
# equivalent to:
# result[0] = 2595 * np.log10(1 + 100 / 700)
# result[1] = 2595 * np.log10(1 + 500 / 700)
# result[2] = 2595 * np.log10(1 + 1000 / 700)
# result[3] = 2595 * np.log10(1 + 4000 / 700)
# in the end result has the same shape as the input
```

It even works for scalars:
```python
f = 1000
result = 2595 * np.log10(1 + f / 700)  # returns a single number
```

That's why we use `np.log10` rather than `math.log10` — `math.log10` only handles scalars.

In [ ]:
def hz_to_mel(f):
    # ✏️ TODO: apply the formula 2595 * log10(1 + f/700)
    # Hint: use np.log10, which supports scalars and arrays
    raise NotImplementedError("TODO: return 2595 * np.log10(1 + f / 700)")

In [ ]:
try:
    assert abs(hz_to_mel(700) - 781.2) < 0.5, f"hz_to_mel(700) = {hz_to_mel(700):.2f}"
    assert hz_to_mel(0) == 0.0, "hz_to_mel(0) should be 0"
    result = hz_to_mel(np.array([0, 700, 8000]))
    assert result.shape == (3,), "array input should return the same shape"
    print(f"hz_to_mel(700) = {hz_to_mel(700):.2f}  ✅")
    print(f"hz_to_mel([0, 700, 8000]) = {hz_to_mel(np.array([0, 700, 8000]))}  ✅")
except (NotImplementedError, TypeError):
    print("⬜ Finish the hz_to_mel exercise above first, then run this cell")

## 5. ✏️ Implement `mel_filterbank(n_mels, n_fft, sr)`

**Line of reasoning**:
1. Take `n_mels + 2` evenly spaced points in the mel domain (including both ends, 0 and `hz_to_mel(sr/2)`), with `np.linspace`
2. Convert every mel point back to Hz: `700 * (10**(mel_points / 2595) - 1)`
3. Map Hz to FFT bin indices: `np.floor(hz_points / (sr / n_fft) * ... )` — the exact formula is `np.floor((n_fft + 1) * hz_points / sr).astype(int)`
4. For each filter `m` (0 to n_mels-1), build the rising/falling ramps over `[bin[m], bin[m+1], bin[m+2]]`:
   - rising edge: `(k - bin[m]) / (bin[m+1] - bin[m])` for k in `[bin[m], bin[m+1]]`
   - falling edge: `(bin[m+2] - k) / (bin[m+2] - bin[m+1])` for k in `[bin[m+1], bin[m+2]]`

**Reference input and output**:
- `mel_filterbank(40, 512, 16000).shape` → `(40, 257)`
- every element `>= 0`, and each row peaks at 1
- `mel_filterbank(40, 512, 16000)[0, :10]`: the first few bins have nonzero weights (filter 0 covers the low frequencies)

> 💡 Need a hint? Once you've done the exercise you can look at the reference implementation in the `solutions/` folder.


### What an FFT bin is, and frequency resolution

You met the FFT around L35 and L37, but here we need to revisit one key idea: the **FFT bin**.

When we take the FFT of $N$ samples (at sample rate sr Hz), we get $N$ frequency-domain values, called **bins** (buckets). Each bin stands for one frequency:
- Bin 0 ← frequency 0 Hz
- Bin 1 ← frequency sr/N Hz
- Bin 2 ← frequency 2·sr/N Hz
- ...
- Bin k ← frequency k·sr/N Hz

**Frequency resolution** = sr/N Hz/bin, the frequency spacing between two neighboring bins.

For example: sr=16000 Hz, N=512 (a common STFT window length)
$$\text{frequency resolution} = \frac{16000}{512} = 31.25 \text{ Hz/bin}$$

So:
- Bin 0 ← 0 Hz
- Bin 1 ← 31.25 Hz
- Bin 10 ← 312.5 Hz
- ...

Because of the symmetry of real signals, we only need the first N//2+1 bins (from 0 Hz to sr/2 Hz); this is called the **one-sided spectrum**.

**Why convert frequencies to bin indices?** Because the FFT output is discrete: we only get bins at integer positions. The center frequency of a mel filter usually doesn't land exactly on a bin, so we need the corresponding bin index to look up the weights.

In [ ]:
# ✏️ Steps 1-3: mel frequency conversion — map Hz to the mel domain, back to Hz, then to FFT bin indices
def mel_filterbank_steps123(n_mels, n_fft, sr):
    """Returns (mel_min, mel_max, mel_points, hz_points, bin_idx) for debugging."""
    mel_min = 0.0
    # ✏️ TODO 1: mel_max = hz_to_mel(sr / 2)
    # ✏️ TODO 2: mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    # ✏️ TODO 3a: hz_points = 700 * (10 ** (mel_points / 2595) - 1)
    # ✏️ TODO 3b: bin_idx = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    raise NotImplementedError("TODO 1-3: delete this line once mel_max / mel_points / hz_points / bin_idx are done")
    return mel_min, mel_max, mel_points, hz_points, bin_idx

try:
    _, _, mpts, hpts, bidx = mel_filterbank_steps123(26, 512, 16000)
    assert mpts is not None, '⬜ mel_points not implemented'
    assert len(mpts) == 28, f'mel_points should have length 28 (n_mels+2), got {len(mpts)}'
    assert abs(hpts[0]) < 1.0, f'the lowest Hz point should be close to 0, got {hpts[0]:.1f}'
    assert abs(hpts[-1] - 8000) < 10, f'the highest Hz point should be close to 8000, got {hpts[-1]:.1f}'
    print(f'✅ Steps 1-3 pass: first and last bin_idx = {bidx[0]}, {bidx[-1]}')
except (NotImplementedError, TypeError):
    print('⬜ mel_filterbank_steps123 not implemented yet')


### Deriving the Hz → FFT bin index formula

We know the frequency each bin stands for is:
$$f_{\text{bin}} = k \cdot \frac{\text{sr}}{n\_\text{fft}}$$

where $k$ is the bin index. Solve for $k$:
$$k = f_{\text{hz}} \cdot \frac{n\_\text{fft}}{\text{sr}}$$

That usually gives a floating-point number, and we need an integer bin. So we round down with `np.floor`:
$$k_{\text{rounded}} = \left\lfloor f_{\text{hz}} \cdot \frac{n\_\text{fft}}{\text{sr}} \right\rfloor$$

**Why does the code use `(n_fft + 1)` instead of `n_fft`?**

This is a detail. The standard formula would be `n_fft / sr * hz`. But here we use **a variant for mapping mel centers to bins**: in some implementations (including this lesson's), a slightly adjusted factor is used to make the mapping of the frequency range more even. The choice of `(n_fft + 1)` comes from:
- physically, we have n_fft//2 + 1 real bins (0 to sr/2)
- that amounts to dividing the whole frequency range into n_fft//2 segments (because of the symmetry)
- using (n_fft+1) in the mapping makes the filters' edge handling more symmetric

A concrete example: sr=16000, n_fft=512
- frequency resolution (standard) = 16000 / 512 = 31.25 Hz/bin
- if a mel center is at 1000 Hz:
  - standard formula: bin = 1000 * 512 / 16000 = 32 (rounded)
  - this lesson's formula: bin = floor((512+1) * 1000 / 16000) = floor(32.0625) = 32 ✓

In most cases the two give the same result; the difference is in edge handling. Production code (aurora.audio.mel, in L47) uses finer **continuous bins** (fractional bins), but the principle is the same.

In [ ]:
# ✏️ Step 4: the triangular filter matrix — fill fb using bin_idx from steps 1-3
def mel_filterbank(n_mels, n_fft, sr):
    """The full mel filterbank; returns a triangular matrix of shape (n_mels, n_fft//2+1)."""
    mel_min = 0.0
    mel_max = hz_to_mel(sr / 2)
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_points = 700 * (10 ** (mel_points / 2595) - 1)
    bin_idx = np.floor((n_fft + 1) * hz_points / sr).astype(int)

    n_bins = n_fft // 2 + 1
    fb = np.zeros((n_mels, n_bins))
    # ✏️ TODO: for filter m, lo=bin_idx[m], ctr=bin_idx[m+1], hi=bin_idx[m+2]
    #   rising edge [lo, ctr): fb[m, k] = (k - lo) / (ctr - lo)
    #   falling edge [ctr, hi): fb[m, k] = (hi - k) / (hi - ctr)
    raise NotImplementedError("TODO step 4: delete this line once fb is filled with triangular weights")
    return fb

try:
    fb = mel_filterbank(26, 512, 16000)
    assert fb.shape == (26, 257), f'shape should be (26, 257), got {fb.shape}'
    assert (fb >= 0).all(), 'filter values should be non-negative'
    assert fb.max() > 0, 'fb is all zeros: step 4 not implemented yet'
    print(f'✅ mel_filterbank has the right shape: {fb.shape}, max value {fb.max():.4f}')
except (NotImplementedError, TypeError):
    print('⬜ mel_filterbank step 4 not implemented yet')


In [ ]:
try:
    fb = mel_filterbank(40, 512, 16000)
    assert fb.shape == (40, 257), f"wrong shape: {fb.shape}"
    assert np.all(fb >= 0), "filter weights should not be negative"
    assert np.allclose(fb.max(axis=1), 1.0), "each filter should peak at 1"
    print(f"filterbank shape: {fb.shape}  ✅")
    print(f"all weights >= 0: {np.all(fb >= 0)}  ✅")
    print(f"every row peaks at 1: {np.allclose(fb.max(axis=1), 1.0)}  ✅")
except (NotImplementedError, TypeError):
    print('⬜ Finish the mel_filterbank exercise above first, then run this cell')

## 6. Parameter experiment: visualizing the filterbank

Change the parameters and watch the effect:
- `n_mels=40` (standard ASR) vs `n_mels=80` (high resolution): twice as many filters, richer high-frequency detail
- `sr=16000` vs `sr=8000`: halving the sample rate drops the highest covered frequency from 8000 Hz to 4000 Hz, and the filters crowd toward the low frequencies
- Plot 1 (linear Hz axis): low-frequency filters narrow and dense, high-frequency filters wide and sparse — the logarithmic character
- Plot 2 (mel axis): all filters equally wide, evenly spread

In [ ]:
try:
    fb40 = mel_filterbank(40, 512, 16000)
    n_bins = 512 // 2 + 1
    freq_axis = np.linspace(0, 16000 / 2, n_bins)
    mel_axis = hz_to_mel(freq_axis)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Plot 1: linear Hz axis
    for m in range(40):
        axes[0].plot(freq_axis, fb40[m], alpha=0.5, linewidth=0.8)
    axes[0].set_title('40 mel filters (linear Hz axis)')
    axes[0].set_xlabel('frequency (Hz)')
    axes[0].set_ylabel('weight')
    axes[0].set_xlim(0, 8000)

    # Plot 2: mel axis
    for m in range(40):
        axes[1].plot(mel_axis, fb40[m], alpha=0.5, linewidth=0.8)
    axes[1].set_title('40 mel filters (mel axis)')
    axes[1].set_xlabel('frequency (mel)')
    axes[1].set_ylabel('weight')

    plt.suptitle('n_mels=40, n_fft=512, sr=16000', fontsize=12)
    plt.tight_layout()
    plt.show()
    print("Left: dense at low frequencies / sparse at high → right: evenly spread — the logarithmic character of the mel scale")
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO exercises above first, then run this cell')

## 3a. Drawing exercise: the three anchor points of a triangular filter

On paper, draw one triangular filter, mark its three anchor points left / center / right, then write down the slopes of the rising and falling edges.
If you get stuck, remember that `hz_to_mel` is only a change of coordinates; the actual shape comes from three neighboring center points.

## Wrap-up

This lesson implemented `hz_to_mel()` and `mel_filterbank()`: the first squeezes linear Hz onto mel coordinates that are closer to how we hear; the second cuts that axis into a triangular weight matrix of shape `(n_mels, n_fft//2+1)`. Multiply it onto an STFT power spectrum and one frame of spectrum collapses into one `(n_mels,)` vector of mel energies.

> ⚠️ **This lesson uses the "integer bin index" version of the triangular filters** (easy to compute by hand); the production `aurora.audio.mel.mel_filterbank()` builds its triangles from "continuous frequencies (fractional bins)" — exactly the same idea, but the weights aren't identical point for point (the edge bins differ slightly). Both are standard practice. The next lesson (L47) wires this filterbank back into the whole pipeline — STFT → power spectrum → mel filtering → log compression — where you implement `log_mel_spectrogram` yourself and match it against aurora's reference output.

## 6.5 Integer bins vs fractional bins: why is "this lesson the simplified version"?

You may have noticed that the wrap-up says "this lesson uses the integer bin index version". What does that mean?

**The integer-bin version (this lesson)**:
- each mel center frequency is mapped to an **integer bin** index (rounded down)
- the triangular filters' anchor points sit at integer bin positions
- the weights are computed at integer bin boundaries, so "jumps" can appear

Example (sr=16000, n_fft=512): if a filter's center should be at 1234 Hz, it maps to bin = floor(513 * 1234 / 16000) = bin 39, while the true position is bin 1234 / 31.25 ≈ 39.49. In the integer-bin version we throw away the fractional part, about 0.49.

**The fractional-bin version (production, aurora in L47)**:
- every frequency keeps a **floating-point bin** position
- the triangular filters' anchor points are represented exactly as fractional bins
- the weights vary smoothly across fractional bin positions, with no jumps

**How big is the effect?**
In most cases (with n_fft large enough, such as 512 or 1024), the difference between the two is small (< 1%). You only see a clear difference when:
- n_fft is very small (< 256)
- a frequency lands right near a bin boundary

**Why does this lesson use the integer version?**
1. It's easy to compute and check by hand (integers are easier than floats)
2. The point of the lesson is the idea "even in the mel domain → uneven in the Hz domain → matches the cochlea", not fine-tuning precision
3. It's friendlier for beginners

**The trade-off**: precision for readability. In real applications (speech recognition, music analysis), fractional bins work better, but for understanding the core principle, integer bins are enough.

## ✏️ Whiteboard challenge: the mel scale by hand (aim for 10 minutes)

Cover the screen and answer on paper:

**Question 1**: convert these Hz → mel by hand:
- 100 Hz → ? mel
- 1000 Hz → ? mel (the reference point)
- 8000 Hz → ? mel

> 💡 Hint: you may use a log table or a simple calculator. The point is to understand the formula mel=2595·log₁₀(1+f/700), not to compute logarithms in your head.
> If you need log₁₀ values for reference, see the table below.

**Question 2**: what is the inverse formula `mel_to_hz(m)`?
(Hint: solve mel = 2595·log₁₀(1+f/700) for f)
Check: mel_to_hz(hz_to_mel(440)) ≈ 440

**Question 3**: `n_mels=40, n_fft=512, sr=16000`:
- highest mel frequency = hz_to_mel(sr/2) = hz_to_mel(8000) = ?
- take 42 evenly spaced points in the mel domain (including both ends); the spacing between neighbors = ? mel

**Question 4**: what is the shape of the triangular filterbank matrix `fb`?
(n_mels=40, n_fft=512 → n_bins=n_fft//2+1=?)

**Question 5**: why do we use **matrix multiplication** rather than element-wise multiplication when applying the filterbank matrix to the power spectrum?
(Hint: the power spectrum has shape=(n_bins,), fb has shape=(n_mels, n_bins); how do you get (n_mels,)?)

Once you've worked it out, run the cell below to check your answers.

### log₁₀ reference table (if you need it)

| Value | log₁₀ |
|------|-------|
| 1.0 | 0.0000 |
| 1.1429 (100/700+1) | 0.0582 |
| 2.4286 (1000/700+1) | 0.3853 |
| 12.4286 (8000/700+1) | 1.0944 |
| 2.5 (1000/400) | 0.3979 |

### Extra: deriving the slope formulas of a triangular filter

Why is the rising edge `(k - lo) / (ctr - lo)`? It's the **linear interpolation** formula.

Draw a straight line from the point (lo, 0) to the point (ctr, 1) and find its value at position k:
$$y(k) = 0 + \frac{k - lo}{ctr - lo} \cdot (1 - 0) = \frac{k - lo}{ctr - lo}$$

This is geometry from school: similar triangles between two points on a line.

Likewise, the falling edge goes from (ctr, 1) to (hi, 0):
$$y(k) = 1 + \frac{k - ctr}{hi - ctr} \cdot (0 - 1) = 1 - \frac{k - ctr}{hi - ctr} = \frac{hi - k}{hi - ctr}$$

**The intuition**:
- the numerator `(k - lo)` or `(hi - k)` is "the distance from the current position to the starting point"
- the denominator `(ctr - lo)` or `(hi - ctr)` is "the total length of the rising/falling edge"
- the ratio is "how far along you are" (from 0 to 1)

That's exactly what you see when you draw the triangle on paper.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: Hz → mel by hand
def _ref_hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

mel_100  = _ref_hz_to_mel(100)
mel_1000 = _ref_hz_to_mel(1000)
mel_8000 = _ref_hz_to_mel(8000)

assert abs(mel_100 - 150.5) < 1.0
assert abs(mel_1000 - 999.9) < 1.0
assert abs(mel_8000 - 2840.0) < 5.0
print(f"Q1 ✅  100→{mel_100:.1f} mel, 1000→{mel_1000:.1f} mel, 8000→{mel_8000:.1f} mel")

# Q2: check the inverse mel_to_hz
def _ref_mel_to_hz(m):
    return 700 * (10 ** (m / 2595) - 1)

f_roundtrip = _ref_mel_to_hz(_ref_hz_to_mel(440))
assert abs(f_roundtrip - 440) < 1e-8
print(f"Q2 ✅  mel_to_hz(hz_to_mel(440)) = {f_roundtrip:.10f} ≈ 440 Hz")

try:
    from aurora.audio.mel import mel_to_hz as aurora_mel_to_hz
    print(f"    Aurora mel_to_hz(hz_to_mel(440)) = {aurora_mel_to_hz(_ref_hz_to_mel(440)):.6f}")
except (ImportError, AttributeError):
    pass

# Q3: n_mels=40, n_fft=512, sr=16000
n_mels, n_fft, sr = 40, 512, 16000
mel_max = _ref_hz_to_mel(sr / 2)
n_center = n_mels + 2
mel_spacing = mel_max / (n_center - 1)
assert abs(mel_max - 2840.0) < 5.0
print(f"Q3 ✅  hz_to_mel(8000)={mel_max:.1f} mel, spacing of the 42 points={mel_spacing:.1f} mel")

# Q4: filterbank shape
n_bins = n_fft // 2 + 1
assert n_bins == 257
try:
    fb = mel_filterbank(n_mels, n_fft, sr)
    assert fb.shape == (n_mels, n_bins), f"wrong shape: {fb.shape}"
    print(f"Q4 ✅  fb.shape = {fb.shape} ({n_mels} filters × {n_bins} FFT bins)")
except (NotImplementedError, NameError):
    print(f"Q4 ✅  n_bins={n_bins}, fb.shape=({n_mels},{n_bins}) (mel_filterbank still to be implemented)")

# Q5: matrix multiplication vs element-wise
# power_spectrum shape=(n_bins,), fb shape=(n_mels, n_bins)
# mel_energy = fb @ power  → (n_mels, n_bins) @ (n_bins,) = (n_mels,)
power = np.ones(n_bins)
fb_ref = np.ones((n_mels, n_bins)) / n_bins  # dummy uniform filterbank
mel_energy = fb_ref @ power
assert mel_energy.shape == (n_mels,)
print(f"Q5 ✅  fb({n_mels},{n_bins}) @ power({n_bins},) = mel_energy({n_mels},) (matrix multiplication reduces the dimension)")
print("\n🎉 Mel scale whiteboard challenge passed!")

In [ ]:
try:
    # ✏️ Self-review for this lesson
    l46_review = {
        "mel_formula":             None,  # remember mel=2595·log₁₀(1+f/700) and its inverse? True/False
        "hz_to_mel_impl":          None,  # hz_to_mel implemented and passes the hz_to_mel(700)≈781 assertion? True/False
        "filterbank_impl":         None,  # mel_filterbank shape=(n_mels,n_fft//2+1) passes? True/False
        "log_perception":          None,  # understand the ear's log perception → evenly spaced mel ≈ even spread along the cochlea? True/False
        "whiteboard_passed":       None,  # whiteboard challenge done on paper? True/False
    }

    unfilled = [k for k, v in l46_review.items() if v is None]
    assert not unfilled, f'not filled in yet: {unfilled}'
    weak = [k for k, v in l46_review.items() if v is False]
    if weak:
        print(f'⚠️  needs more work: {weak}')
    else:
        print('✅ L46 fully cleared! On to L47: writing the mel filterbank')
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO exercises above first, then run this cell')

---

→ **Next lesson** · [L47 · Writing the mel filterbank](L47_mel_implement.ipynb)

> Next lesson covers **writing the mel filterbank** into a full log-mel pipeline: log_mel_spectrogram from formula to NumPy, matched against the repo's output.